In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
keras.config.set_image_data_format("channels_first")

from agx_core.models.ra.encoder import Encoder
from agx_core.models.ra.decoder import Decoder
from agx_torch.models.ra.model import ReversedAutoencoder

In [ ]:
import keras
import torch
from torchvision.models import mobilenet_v3_small

keras.backend.clear_session()

model = mobilenet_v3_small(weights="DEFAULT")
feature_layers = list(model.get_submodule("features").children())
feature_layers.append(keras.layers.AveragePooling2D(2))


# x = keras.random.normal((1, 3, 224, 224)).cpu()
# for idx, layer in enumerate(feature_layers):
#     x = layer(x)
#     print(idx, x.shape)

# 0 torch.Size([1, 16, 112, 112])
# 1 torch.Size([1, 16, 56, 56])
# 2 torch.Size([1, 24, 28, 28])
# 3 torch.Size([1, 24, 28, 28])
# 4 torch.Size([1, 40, 14, 14])
# 5 torch.Size([1, 40, 14, 14])
# 6 torch.Size([1, 40, 14, 14])
# 7 torch.Size([1, 48, 14, 14])
# 8 torch.Size([1, 48, 14, 14])
# 9 torch.Size([1, 96, 7, 7])
# 10 torch.Size([1, 96, 7, 7])
# 11 torch.Size([1, 96, 7, 7])
# 12 torch.Size([1, 576, 7, 7])
# 13 torch.Size([1, 576, 3, 3])


tap_indices = [0, 1, 2, 4, 9, 12, 14]
grouped = [feature_layers[i:j] for i, j in zip(tap_indices, tap_indices[1:])]

backbone = []
for group in grouped:
    if len(group) == 1:
        backbone.append(keras.layers.TorchModuleWrapper(group[0]))
    else:
        backbone.append(keras.layers.TorchModuleWrapper(torch.nn.Sequential(*group)))

# x = keras.random.normal((1, 3, 224, 224)).cuda()
# for idx, layer in enumerate(backbone):
#     x = layer(x)
#     print(idx, x.shape)

# 0 torch.Size([1, 16, 112, 112])
# 1 torch.Size([1, 16, 56, 56])
# 2 torch.Size([1, 24, 28, 28])
# 3 torch.Size([1, 48, 14, 14])
# 4 torch.Size([1, 96, 7, 7])
# 5 torch.Size([1, 576, 3, 3])

In [ ]:
import keras

from typing import Optional, Sequence

from agx_core.models.ra.base import BaseEncoder
from agx_core.models.ra.layers import Split, _axis_channel


class NewEncoder(BaseEncoder):

    def __init__(
        self,
        latent_size: int,
        backbone: Sequence[keras.layers.Layer],
        name="mbnetv3",
        **kwargs,
    ):
        super(NewEncoder, self).__init__(latent_size, name=name, **kwargs)

        self.backbone = backbone
        self.conv: keras.layers.Conv2D = None
        self.gray2rgb = keras.Sequential(
            [
                keras.layers.Conv2D(
                    3, kernel_size=(3, 3), strides=(1, 1), padding="same"
                ),
                keras.layers.LayerNormalization(),
                keras.layers.Activation("tanh"),
            ]
        )
        self.concat = keras.layers.Concatenate(_axis_channel())
        self.split = Split(2, axis=_axis_channel())

    def build(self, input_shape: Sequence[Sequence[int]]):
        x_shape, c_shape = input_shape

        if keras.config.image_data_format() == "channels_last":
            spatial = slice(1, 3)
        else:
            spatial = slice(2, None)

        self.gray2rgb.build(x_shape)

        for block in self.backbone:
            h, w = x_shape[spatial]
            block.build(x_shape)
            x_shape = list(x_shape)
            x_shape[spatial] = [h // 2, w // 2]

        resolution = x_shape[spatial]
        if keras.config.image_data_format() == "channels_last":
            x_shape = (x_shape[0], *resolution, 576)
        else:
            x_shape = (x_shape[0], 576, *resolution)

        # Map downsampled spatial features directly to 1x1 latent dimensions
        self.conv = keras.layers.Conv2D(
            2 * self.latent_size,
            kernel_size=resolution,
            padding="valid",
        )

        self.concat.build([x_shape, c_shape])
        x_shape = self.concat.compute_output_shape([x_shape, c_shape])

        self.conv.build(x_shape)
        x_shape = self.conv.compute_output_shape(x_shape)
        self.split.build(x_shape)

        super(NewEncoder, self).build(input_shape)

    def train_backbone(self, train: bool):
        for layer in self.backbone:
            layer.trainable = train

    def call(
        self,
        inputs: Sequence[keras.KerasTensor],
        training: Optional[bool] = None,
    ) -> Sequence[keras.KerasTensor]:
        x, c = inputs

        x = self.gray2rgb(x, training=training)
        embeddings = []
        for layer in self.backbone:
            x = layer(x, training=training)
            embeddings.append(x)

        x = self.concat([x, c])
        x = self.conv(x)

        mean, logvar = self.split(x)
        return (mean, logvar), embeddings

    def get_config(self):
        config = super().get_config()
        config.update(dict(backbone=keras.saving.serialize_keras_object(self.backbone)))
        return config

In [ ]:
# from agx_core.models.ra.encoder import Encoder
# enc = Encoder()

enc = NewEncoder(512, backbone)

img_size = 224
res = img_size // 2 ** len(enc.backbone)

img_shape = (None, 1, img_size, img_size)
cond_shape = (None, 1, res, res)

dec = Decoder([576, 96, 48, 24, 16, 16], target_shape=img_shape[1:])

ra = ReversedAutoencoder(enc, dec)
ra.build([img_shape, cond_shape])
ra.summary()

In [ ]:
import torch
import numpy as np
import albumentations as A

from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from albumentations.pytorch import ToTensorV2


class UnlabeledImageDataset(Dataset):
    def __init__(self, root_dir: Path, cond_shape: Sequence[int], transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.cond_shape = cond_shape
        # Get list of all image file names in the folder
        self.image_files = list(root_dir.glob("*.bmp"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(img_name).convert("L")

        if self.transform:
            image = self.transform(image=np.array(image))
            image = image["image"]

        condition = np.ones(self.cond_shape, dtype=np.int64)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        image = image.to(device)
        return (image, torch.tensor(condition, device=device)), image


train_path = Path("../data/products/LaTuaPastaGlassJars/Clean/train/")
valid_path = Path("../data/products/LaTuaPastaGlassJars/Clean/val/")

tx_train = A.Compose(
    [
        A.InvertImg(1),
        A.Resize(img_size, img_size),
        A.HorizontalFlip(0.5),
        A.Affine(scale=(0.9, 0.95), rotate=(15, 90), shear=(-2, 2), p=0.5),
        A.RandomRotate90(0.5),
        A.RandomBrightnessContrast(
            brightness_limit=0.2, contrast_limit=(-0.5, 0.5), p=0.5
        ),
        A.GaussianBlur(blur_limit=(1, 3), p=0.3),
        A.Normalize(mean=[0.7], std=[0.4]),
        ToTensorV2(),
    ]
)

tx_valid = A.Compose(
    [
        A.InvertImg(1),
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.7], std=[0.4]),
        ToTensorV2(),
    ]
)

ds_train = UnlabeledImageDataset(
    train_path, transform=tx_train, cond_shape=cond_shape[1:]
)
ds_valid = UnlabeledImageDataset(
    valid_path, transform=tx_valid, cond_shape=cond_shape[1:]
)

loader_train = DataLoader(ds_train, batch_size=16, shuffle=True)
loader_valid = DataLoader(ds_valid, batch_size=16)

In [ ]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# for idx, ax in enumerate(axes.flat):
#     X, y = ds_train[idx]
#     ax.imshow(X[0][0], cmap="gray")
#     ax.set_title(f"Image {idx + 1}")
#     ax.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint

from agx_core.models.ra.optimizer import RAOptimizer
from agx_core.models.ra.callbacks import AdversarialEquilibriumCallback, BackboneThawCallback


optimizer = RAOptimizer(Adam(learning_rate=1e-9), Adam(learning_rate=6e-6))

callbacks = [
    AdversarialEquilibriumCallback(),
    BackboneThawCallback(),
    ModelCheckpoint(
        filepath="ra_mbnetv3.model.keras",
        monitor="val_loss_rec",
        mode="min",
        save_best_only=True,
    )
]

ra.compile(optimizer)

In [ ]:
history2 = ra.fit(loader_train, validation_data=loader_valid, epochs=2000, initial_epoch=1000, callbacks=callbacks, verbose=2)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    (I, C), y = ds_valid[idx]
    rec = ra([I[np.newaxis, ...], C[np.newaxis, ...]])
    ax.imshow(rec.cpu().detach().numpy()[0][0], cmap="gray")
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

fig, ax = plt.subplots(figsize=(21, 14))

df = pd.DataFrame.from_dict(history2.history)
# Adjust val_loss_embed, forgot to update validation scaling
df["val_loss_embed"] /= (ra.scale ** 3)
df[["loss_rec", "val_loss_rec"]].plot(ax=ax)

In [ ]:
# device = torch.device("cuda")
# ra.eval()
# ra = ra.to(device)
# I, C = torch.rand((1, *img_shape[1:]), device=device), torch.rand(
#     (1, *cond_shape[1:]), device=device
# )

# torch.onnx.export(
#     ra.encoder,
#     ([I, C],),
#     "model.onnx",
#     input_names=["image", "condition"],
#     output_names=["reconstruction"],
# )